# Map flux
## Make the same maps as the one presented in https://arxiv.org/pdf/2104.06855.pdf

### Import modules

In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import Colormap

/home/tomas/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


### User parameters

In [2]:
incl = 48 * np.pi / 180.0
Nx = 32
f_Ny = 0.15 # Nyquist frequency (< 0.6)
#co = sqrt(8.d0*log(2.d0))       ! difference between sigma and theta!, theta = sigma*co

### Define some constants

In [3]:
h = 6.6260755e-27
eV = 1.602176e-12
kpc = 3.0856e21
distance = 3.8e3
convert = 206265e3/(distance*kpc) # image to angular size (mas)
flux = 1.e26*(h/eV)/(4.0*np.pi*(distance*kpc)**2) # L[erg/s/erg] -> F[mJy]

### Define some useful functions

In [4]:
lumRIAF = np.loadtxt('../RIAF_radproc/build/src/adaf/lumThermal.dat').transpose()

nu = lumRIAF[0][16]
fluxTot2 = lumRIAF[8][16]

FileNotFoundError: ../RIAF_radproc/build/src/adaf/lumThermal.dat not found.

In [5]:
def read_SEDsyn_cuts(filename='../Code/SEDe_syn_cuts_ord.dat'):
    fname = filename
    #fname format: Esy, Lsy*Esy, Labssy, l
    data = np.loadtxt(fname).transpose()
    NSED_sy = 0
    value = data[3][0]
    while data[3][NSED_sy] == value:
        NSED_sy += 1
    Ns = np.size(data[3])//NSED_sy
    print("(Ns, NSED_sy) = (", Ns, ", ", NSED_sy, ")")
    Esy = data[0][:NSED_sy]
    Lsy = np.zeros((Ns, NSED_sy))
    for i in range(Ns):
        Lsy[i] = data[2][i*NSED_sy:(i+1)*NSED_sy] / data[0][:NSED_sy]
    
    return Esy, Lsy

In [6]:
def get_Lsyn_z( Esy, Lsy, nu_target):
    E_nu_target = h * nu_target * 1e9 / eV
    # Step through energies:
    i_nu_target = 0
    while Esy[i_nu_target] < E_nu_target:
        i_nu_target += 1
    
    # Make a linear interpolation
    slope = (Lsy[:, i_nu_target] - Lsy[:, i_nu_target-1]) / (Esy[i_nu_target] - Esy[i_nu_target-1])
    Ltarget = Lsy[:, i_nu_target-1] + slope * (E_nu_target - Esy[i_nu_target-1])
    Ltarget[Ltarget == 0] = 1e-10 # Mock value to avoid dividing by zero
    return Ltarget

In [7]:
def get_z_rz( filename='../Code/quant_FF.dat'):
    # These values are defined in the observer frame
    fname = filename
    data = np.loadtxt(fname, usecols=(0, 2))
    print(data)
    return data[:, 0], data[:, 1]

In [8]:
#Grafico de la luminosidad de brillo bs temperatura
#El cociente entre brillo del jet y contrajet nos da una idea del factor de lorentz del jet
#Las velocidades estimadas por el paper pueden usarse para las velocidades de la vaina y la espina del jet
#Plotear directamente el fmap (estoy ploteando el log de fmap_c, pero puedo plotear solo fmap sin convolucionar, deberia dar mas cercano a lo que produce el modelo, menos difuminado)


In [ ]:
def make_F_map( nu, Nx ):
    Esy, Lsy = read_SEDsyn_cuts()
    L_z = get_Lsyn_z( Esy, Lsy, nu )[:440]
    z, r_z = get_z_rz()
    z = z[:440]
    r_z = r_z[:440]
    dz = z[1] - z[0]
    if dz*np.sin(incl) > 2.0*r_z[-1]*np.cos(incl): #si se cumple para el último se cumple para todos
    #if 0: #Para no entrar a lateral
        x = np.linspace( 0, r_z[-1], Nx) * convert
        dx = x[1] - x[0]
        print('Entramos a lateral_projection')
        y_map = z * np.sin(incl) * convert
        r_z_proj = r_z * convert
        L_map = np.ones(( len( x), len( y_map))) * 0
        for j in range( len( y_map)):
            for i in range( len( x)):
                if x[ i] < r_z_proj[ j]: #Estoy adentro del jet
                    L_map[i,j] = 2.0 * L_z[ j] * np.sqrt( r_z_proj[ j]**2.0 - x[ i]**2.0) / (np.pi * r_z_proj[ j]**2.0)
                    L_map[i,j] = L_map[i,j] * dx
            # Correction factor for better conservation of luminosity
            temp = np.sum( L_map[:,j])
            L_map[:,j] = L_map[:,j] * 0.5*L_z[j] / temp
    else:
        print('Entramos a faceon_projection')
        dy = 0.5 * r_z[0] * np.cos(incl) * convert # delta_y < 2*r_j*cos(i) para todo r_j
        dx = 0.5 * r_z[0] * convert
        y_max = (z[-1]*np.sin(incl) + r_z[-1]*np.cos(incl)) * convert
        x_max = r_z[-1] * convert
        Ny = int(y_max/dy)
        Nx = int(x_max/dx)
        print('Ny = ' + str(Ny))
        print('Nx = ' + str(Nx))
        y_map = np.linspace(0, y_max, Ny )
        x_map = np.linspace(0, x_max, Nx )

        L_map = np.zeros((np.size(x_map), np.size(y_map)))

        x_double = np.ones_like(L_map)
        y_double = np.ones_like(L_map)
        dx = x_map[1] - x_map[0]
        dy = y_map[1] - y_map[0]
        for j in range(len(y_map)):
            x_double[:, j] = x_map
        for i in range(len(x_map)):
            y_double[i, :] = y_map

    ellipses = []
    iy_start = 0
    L_tot = 0.
    for iz in range(np.size(L_z)):
        print(iz)
        center = z[iz] * np.sin(incl) * convert
        a_semiaxis = r_z[iz] * convert
        b_semiaxis = r_z[iz] * np.cos(incl) * convert
        inv_area = np.power(np.pi*a_semiaxis*b_semiaxis, -1)
        # Ecuacion: (x/a)²+((y-z.sin(i))/b)²=1
        ell = []
        for s in np.linspace(0, 2.*np.pi, 30):
            ell.append([a_semiaxis*np.cos(s), b_semiaxis*np.sin(s)+center])
        ellipses.append(ell)
        L_local = np.where( np.square(x_double/a_semiaxis) + np.square((y_double-center)/b_semiaxis) < 1.,
                 1, 0) * L_z[iz] * min(dy*dx*inv_area, 0.5) #Matriz de intensidad para cada pixel
        renorm = 1./(2.*np.sum(L_local)/L_z[iz]) # Renormalization factor to conserve flux
        L_local = L_local*renorm
        L_map += L_local #COnsidera la superposicion de elipses de dos segmentos
        L_local = np.sum(L_local)
        L_tot += L_local
    print( 'L_proj/L_pre-proj = ' + str(2*L_tot/np.sum(L_z)) )

    return x_map, y_map, L_map

In [10]:
def Flux_mirror_map( x_map, L_map ):
    Nx = len(x_map)
    x_map22 = np.ones( 2*Nx )
    F_map = np.ones( (2*len(L_map[:, 0]), len(L_map[0, :])) )

    x_map22[:Nx] = -np.flip(x_map)
    x_map22[Nx:] = x_map
    F_map[:Nx, :] = np.flip(L_map, axis=0) * flux
    F_map[Nx:, :] = L_map*flux
    return x_map22, F_map

In [11]:
def Convolve_map( x_map, x_map22, y_map, F_map, sigma_x, sigma_y ):    
    # The number of points in the convolved map depends on the beam size and the size of the original map.

    # Extremes of the convolved map
    y_min_c = y_map[0] - 4. * sigma_y # 5 beams to the left of the jet base
    y_max_c = y_map[-1] + 4. * sigma_y # 5 beams to the right of the jet end point
    x_min_c = x_map22[0] - 3. * sigma_x # 3 beams above the jet lateral end point
    x_max_c = x_map[-1] + 3. * sigma_x # 3 beams below the jet lateral end point

    # Size of the convolved map
    ancho_x = x_max_c - x_min_c
    ancho_y = y_max_c - y_min_c 

    # Size of each cell of the convolved map
    Dx_c = f_Ny*sigma_x
    Dy_c = f_Ny*sigma_y

    # Number of cells of the convolved map
    Nx_c = int(ancho_x/Dx_c)
    Ny_c = int(ancho_y/Dy_c)

    # Position of the cells of the convolved map
    x_c = np.linspace( x_min_c, x_max_c, Nx_c)
    y_c = np.linspace( y_min_c, y_max_c, Ny_c)

    print('Dimension of the convolution grid: ' + str(Nx_c) + 'x' + str(Ny_c))
    
    F_map_c = np.ones((Nx_c, Ny_c))

    w_x = 0.5/sigma_x**2 # weight for x distance
    w_y = 0.5/sigma_y**2 # weight for y distance

    x_double = np.ones_like(F_map)
    y_double = np.ones_like(F_map)
    for j in range(len(y_map)):
        x_double[:, j] = x_map22
    for i in range(len(x_map22)):
        y_double[i, :] = y_map

    for i in range(Nx_c):
        for j in range(Ny_c):
            # Gaussian: (1/(2pi*sigma_x*sigma_y)) * exp(-((x-x_c)²/2.sigma_x² + (y-y_c)²/2.sigma_y²))
            gaussian = np.exp(-(np.square(x_double-x_c[i])*w_x + np.square(y_double-y_c[j])*w_y))
            F_map_c[i, j] = np.sum(F_map*gaussian)
            
    S_pre = np.sum(F_map)
    print('Pre-conv: S(GHz) = ' + str(S_pre) + ' mJy')
    # Gaussian beam -> exp(...)/(sigma*sqrt(2*pi)), both in x and y
    A_beam = 2 * np.pi * sigma_x * sigma_y     # = Gaussian normalization factor 
    A_cell = Dx_c * Dy_c              # cell sample size
    mJy_beam = A_cell/A_beam          # Transform mJy/beam -> mJy (for checks)
    S_post = np.sum(F_map_c) * mJy_beam
    print('Post-conv: S(GHz) = ' + str(S_post) + ' mJy')
    print('post/pre = ' + str(S_post/S_pre) )
    
    return x_c, y_c, F_map_c

In [12]:
def Make_spix_map( x_c, y_c, nu1, F_map1, nu2, F_map2, clip1, clip2):
    spix_map = np.empty((len(x_c), len(y_c))) 
    spix_map[:] = np.nan # Initialize to NaN
    log_nu2nu1 = np.log(nu2/nu1)

    for i in range(len(x_c)):
        for j in range(len(y_c)):
            print(F_map1[i,j] > clip1 and F_map2[i,j] > clip2 )
            if ( F_map1[i,j] > clip1 and F_map2[i,j] > clip2 ): 
                   spix_map[i,j] = np.log( F_map2[i,j]/F_map1[i,j] ) / log_nu2nu1
    return spix_map

## Make the intensity map at 43 GHz

In [13]:
nu_43 = 230.0 #43.0 # GHz
sigma_x, sigma_y = 0.01, 0.01 #0.31, 0.31 
x_map, y_map, L_map = make_F_map( nu_43, Nx )
# First mirror and convert to fluxes
x_map22, F_map = Flux_mirror_map( x_map, L_map)
x_c0, y_c0, F_map_c = Convolve_map( x_map, x_map22, y_map, F_map, sigma_x, sigma_y )

/tmp/ipykernel_12190/1725560376.py:4: UserWarning: loadtxt: input contained no data: "./../Code/SEDe_syn_cuts_ord.dat"
  data = np.loadtxt(fname).transpose()


IndexError: index 3 is out of bounds for axis 0 with size 0

In [13]:
#Temperatura de brillo es proporcional a la intensidad especifica
#EN el grafico del paper esta graficado en temperatura de brillo^1/2

In [20]:
from matplotlib.patches import Ellipse
# nice_fonts = {
#         # Use LaTeX to write all text
#         "text.usetex": False,
#         "font.family": 'serif',
#         # Use 10pt font in plots, to match 10pt font in document
#         "axes.labelsize": 16,
#         "font.size": 16,
#         "axes.linewidth":1,
#         "axes.titlesize":20,
#         # Make the legend/label fonts a little smaller
#         "legend.fontsize": 10,
#         "xtick.labelsize": 14,
#         "ytick.labelsize": 14,
# }
# matplotlib.rcParams.update(nice_fonts)


#plt.clf()
with plt.style.context(('classic')):
    fig, ax = plt.subplots(1, 1, figsize=(7.8, 6.6)) # proportion: 2.6/2.2
    #fig, ax = plt.subplots(1, 1) 
    ax.set_xlabel(r"$y~{\rm [mas]}$")
    ax.set_ylabel(r"$x~{\rm [mas]}$")
    ax.set_xlim([-0.05, 0.2])
    ax.set_ylim([-0.05, 0.05])
    palette = plt.cm.afmhot_r
    #cmap.set_label(r"$S_\nu$ [mJy beam$^{-1}$]")
    ticks = list(np.linspace(0.005,10,9))
    CS_conv = ax.contourf( y_c0, x_c0, (F_map_c), cmap=palette, levels = np.linspace(-5, 1, 100) )
    #Agregar contornos a (1, 1.4, 2, 2.8...)*3 mJy beam^-1 --> son 19? aprox
    obs_cont = (np.array([1, 1.4, 2, 2.8, 4, 5.6, 8, 10.2, 16, 20.4, 32, 40.8, 64, 81.6, 128, 163.2, 256])*.025)
    plot_cont = ax.contour(y_c0, x_c0, (F_map_c), linewidths=1, levels=obs_cont, extend="both", colors='dimgrey')

    el = Ellipse((0.15, -0.04), sigma_y, sigma_x, color='grey')
    ax.add_patch(el)

    ax.grid()
    plt.tight_layout()
    fig.colorbar( CS_conv, ticks=ticks ,label=r'$S_\nu~[{\rm Jy~beam}^{-1}]$')

In [21]:
print(np.min(np.log10(F_map_c)), np.max(np.log10(F_map_c)))
print(obs_cont)


-9.230591644656057 0.5850776078210622
[0.025 0.035 0.05  0.07  0.1   0.14  0.2   0.255 0.4   0.51  0.8   1.02
 1.6   2.04  3.2   4.08  6.4  ]


In [22]:
# Save plot
fig.savefig('Map_I_230GHz.pdf', bbox_inches='tight', dpi=300)
fig.savefig('Map_I_230GHz.png', bbox_inches='tight', dpi=100)

## Make the spix map between 22-43 GHz

In [156]:
sigma_x, sigma_y = 1.2, 1.2

nu1 = 22.0 # GHz
x_map, y_map, L_map2 = make_F_map( nu1, Nx )
x_map22, F_map1 = Flux_mirror_map( x_map, L_map2)
x_c, y_c, F_map_c1 = Convolve_map( x_map, x_map22, y_map, F_map1, sigma_x, sigma_y )

nu2 = 43.0 # GHz
x_map, y_map, L_map2 = make_F_map( nu2, Nx )
x_map22, F_map2 = Flux_mirror_map( x_map, L_map2)
x_c, y_c, F_map_c2 = Convolve_map( x_map, x_map22, y_map, F_map2, sigma_x, sigma_y )

(Ns, NSED_sy) = ( 750 ,  150 )
Entramos a faceon_projection
Ny = 9
Nx = 7
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
L_proj/L_pre-proj = 1.0000000000000004
Dimension of the convolution grid: 40x53
Pre-conv: S(GHz) = 1.627086747162803 mJy
Post-conv: S(GHz) = 1.540942980958461 mJy
post/pre = 0.9470564391514138
(Ns, NSED_sy) = ( 750 ,  150 )
Entramos a faceon_projection
Ny = 9
Nx = 7
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72


In [175]:
clip1, clip2 = 0.5, 0.5 # Clip below 10 rms
spix_map = Make_spix_map( x_c, y_c, nu1, F_map_c1, nu2, F_map_c2, clip1, clip2)

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
Fals

In [176]:
plt.clf()
fig, ax = plt.subplots(1, 1, figsize=(8, 3)) # proportion: 16/6
ax.set_xlabel(r"$y$ [mas]")
ax.set_ylabel(r"$x$ [mas]")
ax.set_xlim([-3, 13])
ax.set_ylim([-3, 3])
palette  = plt.cm.jet 
ticks = [-2, -1.5, -1, -0.5, 0, 0.5, 1] 
levels = np.linspace(-0.1, 2, 100)
print(spix_map)
CS_conv = ax.contourf( y_c, x_c, spix_map, cmap=palette)# levels=levels)
# intensity contours at 22 GHz
obs_cont = np.array([1, 1.4, 2, 2.8, 4, 5.6, 8, 10.2, 16, 20.4, 32, 40.8, 64, 81.6, 128, 163.2, 256, 326.4, 512])*1.9 # HACERLOS EN GRIS
plot_cont = ax.contour( y_c, x_c, F_map_c1, levels=obs_cont, extend="both", colors='grey')

el = Ellipse((12, -2.3), sigma_y, sigma_x, color='grey')
ax.add_patch(el)

plt.tight_layout()
fig.colorbar(CS_conv, ticks=ticks)

[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]


In [177]:
# Save plot
fig.savefig('Map_spix_22-43GHz_2_c.pdf', bbox_inches='tight')

## Make the spix map between 43-86 GHz

In [1]:
Nx0 = 64
sigma_x, sigma_y = 0.4, 0.2

nu1 = 43.0 # GHz
x_map, y_map, L_map2 = make_F_map( nu1, Nx0 )
x_map22, F_map1 = Flux_mirror_map( x_map, L_map2)
x_c, y_c, F_map_c1 = Convolve_map( x_map, x_map22, y_map, F_map1, sigma_x, sigma_y )

nu2 = 86.0 # GHz
x_map, y_map, L_map2 = make_F_map( nu2, Nx0 )
x_map22, F_map2 = Flux_mirror_map( x_map, L_map2)
x_c, y_c, F_map_c2 = Convolve_map( x_map, x_map22, y_map, F_map2, sigma_x, sigma_y )

NameError: name 'make_F_map' is not defined

In [79]:
clip1, clip2 = 0.05, 0.05 # Clip below 10 rms
spix_map = Make_spix_map( x_c, y_c, nu1, F_map_c1, nu2, F_map_c2, clip1, clip2)

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
Fals

In [80]:
plt.clf()
fig, ax = plt.subplots(1, 1, figsize=(7.5, 4)) # proportion: 5/2
ax.set_xlabel(r"$y$ [mas]")
ax.set_ylabel(r"$x$ [mas]")
ax.set_xlim([-1, 4])
ax.set_ylim([-1, 1])
palette  = plt.cm.jet 
ticks = [-2, -1.5, -1, -0.5, 0, 0.5, 1] 
levels = np.linspace(-2., 0.5, 100)
CS_conv = ax.contourf( y_c, x_c, spix_map, cmap=palette, levels=levels)

# intensity contours at 43 GHz
obs_cont = np.array([1, 1.4, 2, 2.8, 4, 5.6, 8, 10.2, 16, 20.4, 32, 40.8, 64, 81.6, 128, 163.2, 256, 326.4, 512])*1.9 # HACERLOS EN GRIS
plot_cont = ax.contour( y_c, x_c, F_map_c1, levels=obs_cont, extend="both", colors='grey')

el = Ellipse((3.5, -0.7), sigma_y, sigma_x, color='grey')
ax.add_patch(el)

plt.tight_layout()
fig.colorbar(CS_conv, ticks=ticks)

In [81]:
# Save plot
fig.savefig('Map_spix_43-86GHz_2.pdf', bbox_inches='tight')